# Graphene band structure: from tight binding to Dirac cones

**Professor Nano Labs · Crystals & energy bands**

**Try it live in your browser:** [Graphene band structure lab](https://tonylowgroup.github.io/Professor-Nano-Labs/labs/graphene-bands/)
**Watch first:** [Calculating the band structure of graphene](https://www.youtube.com/watch?v=3b34QM15JTM), then [Deriving the Dirac Hamiltonian in Graphene](https://www.youtube.com/watch?v=TuWvO8oig0E)
**Watch later:** [Calculating electron and hole densities in graphene](https://www.youtube.com/watch?v=wBRvi_NRIog) · [Deriving Berry phase, Berry connection & Berry curvature](https://www.youtube.com/watch?v=6xctcmwR8wo) · [Discovering the relativistic Dirac equation with Paul Dirac and graphene](https://www.youtube.com/watch?v=QYUyAkz7DuA)

The honeycomb lattice has two atoms per cell, A and B. Each A atom has three B neighbours at $\boldsymbol\delta_{1,2}=a(\tfrac12,\pm\tfrac{\sqrt3}{2})$ and $\boldsymbol\delta_3=a(-1,0)$, with $a=0.142$ nm. Nearest-neighbour hopping $-t$ gives the Bloch Hamiltonian
$$H(\mathbf k)=\begin{pmatrix}\Delta&-t\,g(\mathbf k)\\-t\,g^*(\mathbf k)&-\Delta\end{pmatrix},\qquad g(\mathbf k)=\sum_{j=1}^3e^{i\mathbf k\cdot\boldsymbol\delta_j},\qquad E_\pm=\pm\sqrt{\Delta^2+t^2|g|^2},$$
with $t=2.7$ eV and an optional sublattice energy $\pm\Delta$. The three sections match the lab's three tabs. It needs NumPy, SciPy and Matplotlib. **Run all cells from the top.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import ellipk
from scipy.constants import hbar, e, m_e

a, t = 0.142, 2.7                                     # nm, eV
s3 = np.sqrt(3)
DEL = np.array([[a / 2, s3 * a / 2], [a / 2, -s3 * a / 2], [-a, 0]])
KK = 2 * np.pi / (3 * a)
G, K, Kp, M = np.zeros(2), np.array([KK, KK / s3]), np.array([KK, -KK / s3]), np.array([KK, 0])
B1, B2 = KK * np.array([1, s3]), KK * np.array([1, -s3])
cell = 1.5 * s3 * a * a                               # nm^2, two atoms

def g(kx, ky):
    return sum(np.exp(1j * (kx * dx + ky * dy)) for dx, dy in DEL)

def H(kx, ky, D=0.0, tt=t):
    gk = g(kx, ky)
    return np.array([[D, -tt * gk], [-tt * np.conj(gk), -D]])

def bands(kx, ky, D=0.0, tt=t):
    s = np.sqrt(D * D + tt * tt * np.abs(g(kx, ky)) ** 2)
    return -s, s

# the 2x2 diagonalisation agrees with the closed form, and |g|^2 with the video's cosine formula
rng = np.random.default_rng(0)
ks = rng.uniform(-20, 20, size=(200, 2))
print('eigh vs closed form, max difference:', max(np.max(np.abs(np.linalg.eigvalsh(H(*k, 0.3)) - np.array(bands(*k, 0.3)))) for k in ks))
cosform = lambda kx, ky: 3 + 2 * np.cos(s3 * ky * a) + 4 * np.cos(1.5 * kx * a) * np.cos(s3 / 2 * ky * a)
print('|g|^2 vs 3 + 2cos(sqrt3 ky a) + 4cos(3kx a/2)cos(sqrt3 ky a/2):', np.max(np.abs(np.abs(g(ks[:, 0], ks[:, 1])) ** 2 - cosform(ks[:, 0], ks[:, 1]))))
print('|g| at Gamma, K, K\', M:', [round(float(abs(g(*q))), 12) for q in (G, K, Kp, M)])
print('the three phases at K (degrees):', np.round(np.degrees(np.angle(np.exp(1j * DEL @ K))), 6))

## 1 · Tight binding

The bands along Γ → K → M → Γ. At Γ the three hops add up ($|g|=3$, $E=\pm3t$); at M, $|g|=1$ and $E=\pm t$; at K the three phases are 120° apart and cancel, so the bands touch at zero energy.

In [ ]:
path = [G, K, M, G]
seg = [np.linspace(path[i], path[i + 1], 200, endpoint=False) for i in range(3)] + [path[-1][None, :]]
kpath = np.vstack(seg)
dist = np.concatenate([[0], np.cumsum(np.linalg.norm(np.diff(kpath, axis=0), axis=1))])
Em, Ep = bands(kpath[:, 0], kpath[:, 1])
ticks = [0, 200, 400, 600]
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(dist, Em, 'C0', dist, Ep, 'C3')
for i in ticks: ax.axvline(dist[i], color='0.8', lw=0.8)
ax.set_xticks(dist[ticks], ['Γ', 'K', 'M', 'Γ']); ax.set_ylabel('E (eV)'); ax.axhline(0, color='k', lw=0.6)
plt.tight_layout(); plt.show()
print('E at Gamma, K, M:', [tuple(np.round(bands(*q), 4)) for q in (G, K, M)], 'eV')

## 2 · The bands across the Brillouin zone

A contour map of the upper band, and the density of states. The DOS of the nearest-neighbour honeycomb has an exact form in terms of the complete elliptic integral $\mathbb K$; we check a histogram over the Brillouin zone against it. Near zero energy it is the Dirac result $g(E)=2|E|/\pi\hbar^2v_F^2$ (spin and valley included), and it diverges at $E=\pm t$, the van Hove singularities from the saddle points at M.

In [ ]:
R = 1.2 * 2 * KK / s3
x = np.linspace(-R, R, 401)
KX, KY = np.meshgrid(x, x)
fig, axs = plt.subplots(1, 2, figsize=(12, 4.6))
cs = axs[0].contourf(KX, KY, bands(KX, KY)[1], 30, cmap='Oranges')
axs[0].contour(KX, KY, bands(KX, KY)[1], [0.6, 2.7], colors=['k', 'w'], linewidths=1.2)
hexa = np.array([[2 * KK / s3 * np.cos(np.pi / 6 + i * np.pi / 3), 2 * KK / s3 * np.sin(np.pi / 6 + i * np.pi / 3)] for i in range(7)])
axs[0].plot(*hexa.T, 'k', lw=1); axs[0].set_aspect('equal'); axs[0].set(xlabel='kx (1/nm)', ylabel='ky (1/nm)', title='upper band; contours at 0.6 eV (black) and t (white)')

# DOS: histogram over the zone vs the exact formula (Hobson & Nierenberg; Castro Neto et al., RMP 2009)
N = 1000
u, v = np.meshgrid((np.arange(N) + 0.5) / N, (np.arange(N) + 0.5) / N)
kx, ky = u * B1[0] + v * B2[0], u * B1[1] + v * B2[1]
E_all = np.concatenate(bands(kx, ky)).ravel()
dE = 0.02
hist, edges = np.histogram(E_all, bins=np.arange(-8.2, 8.2 + dE, dE))
Ec = (edges[:-1] + edges[1:]) / 2
rho_hist = 2 * hist / (N * N * dE)                          # states per eV per cell, spin included
def rho_exact(E):
    x = np.abs(E) / t
    if x == 0 or x >= 3: return 0.0
    Z0 = (1 + x) ** 2 - (x * x - 1) ** 2 / 4 if x <= 1 else 4 * x
    Z1 = 4 * x if x <= 1 else (1 + x) ** 2 - (x * x - 1) ** 2 / 4
    return 4 / np.pi ** 2 * x / t / np.sqrt(Z0) * ellipk(Z1 / Z0)      # per eV per cell, both spins
rho_ex = np.array([rho_exact(q) for q in Ec])
hv = 1.5 * t * a                                             # hbar v_F in eV nm
axs[1].plot(Ec, rho_hist, color='0.6', lw=1, label='histogram')
axs[1].plot(Ec, rho_ex, 'C2', lw=1.6, label='exact')
axs[1].plot(Ec, 2 * np.abs(Ec) / (np.pi * hv ** 2) * cell, 'k--', lw=1, label='Dirac')
axs[1].set(xlabel='E (eV)', ylabel='states per eV per cell', ylim=(0, 1)); axs[1].legend()
plt.tight_layout(); plt.show()
mask = (np.abs(np.abs(Ec) - t) > 0.2) & (np.abs(Ec) > 0.1) & (np.abs(Ec) < 7.9)
print('states in both bands:', round(float(rho_hist.sum() * dE), 4), ' (exact formula:', round(float(rho_ex.sum() * dE), 3), ')')
print('histogram vs exact, median relative difference away from the singularities:', round(float(np.median(np.abs(rho_hist[mask] / rho_ex[mask] - 1))), 4))

def density(Ef, D=0.0):
    # electrons (+) or holes (-) per nm^2 from neutrality, counting the k-grid states between 0 and E_F
    lo, up = bands(kx, ky, D)
    if Ef >= 0: return 2 * np.sum((up > 0) & (up < Ef)) / (N * N) / cell
    return -2 * np.sum((lo < 0) & (lo > Ef)) / (N * N) / cell
for Ef in (0.3, 1.0, 2.7):
    print(f'E_F = {Ef} eV: n = {density(Ef) * 1e14:.3e} cm^-2 (Dirac: {Ef ** 2 / (np.pi * hv ** 2) * 1e14:.3e})')

## 3 · The Dirac cone

Expanding $g$ near K to first order in $\mathbf q=\mathbf k-\mathbf K$ gives $|g|\approx\tfrac{3a}{2}|\mathbf q|$, so $E=\pm\hbar v_F|\mathbf q|$ with $v_F=3ta/2\hbar$. Below we compare the cone with tight binding, measure the trigonal warping of the constant-energy contours, and compute the Berry phase of the upper band around each valley, $\gamma=-\arg\prod_i\langle u_i|u_{i+1}\rangle$. With no gap it is $\pi$ in both valleys; $+\pi$ and $-\pi$ are the same phase, so its sign means nothing there. A sublattice energy $\Delta$ makes the sign meaningful: $\pm\pi(1-\Delta/E)$, opposite in the two valleys.

In [ ]:
vF = 1.5 * t * e * a * 1e-9 / hbar
print(f'v_F = {vF:.4e} m/s = c / {2.998e8 / vF:.0f};  hbar v_F = {hv:.4f} eV nm')
q = np.linspace(-1.5, 1.5, 301)
fig, axs = plt.subplots(1, 2, figsize=(12, 4.2))
for direction, ls in [((G - K) / np.linalg.norm(G - K), '-'), ((M - K) / np.linalg.norm(M - K), '--')]:
    kk = K[None, :] + q[:, None] * direction[None, :]
    axs[0].plot(q, bands(kk[:, 0], kk[:, 1])[1], 'C3', ls=ls)
axs[0].plot(q, hv * np.abs(q), 'k:', lw=1.4)
axs[0].set(xlabel='q (1/nm), towards Γ (solid) and M (dashed)', ylabel='E (eV)', title='tight binding vs Dirac (dotted)')

def contour(E, valley=K, D=0.0, n=360):
    # radius of the upper-band contour at energy E, along n directions from the valley (first crossing)
    phis = 2 * np.pi * np.arange(n) / n
    out = []
    for p in phis:
        d = np.array([np.cos(p), np.sin(p)])
        qs = np.linspace(0, 0.95 * 2 * KK / s3, 4000)
        Eq = bands(valley[0] + qs * d[0], valley[1] + qs * d[1], D)[1]
        if not np.any(Eq > E):          # no crossing along this ray: the contour is not closed round the valley
            return phis, np.full(n, np.nan)
        i = np.argmax(Eq > E)
        q0, q1 = qs[i - 1], qs[i]
        for _ in range(50):
            qm = (q0 + q1) / 2
            if bands(*(valley + qm * d), D)[1] < E: q0 = qm
            else: q1 = qm
        out.append((q0 + q1) / 2)
    return phis, np.array(out)

for E in (0.1, 1.0):
    phis, r = contour(E)
    axs[1].plot(r * np.cos(phis), r * np.sin(phis), label=f'{E} eV')
    axs[1].plot(E / hv * np.cos(phis), E / hv * np.sin(phis), 'k:', lw=1)
    print(f'E = {E} eV: radius {r.min():.4f}..{r.max():.4f} 1/nm (Dirac {E / hv:.4f}); max/min - 1 = {100 * (r.max() / r.min() - 1):.1f}%; mean {100 * (r.mean() * hv / E - 1):+.2f}%')
axs[1].set_aspect('equal'); axs[1].legend(); axs[1].set(xlabel='qx', ylabel='qy', title='contours round K, with Dirac circles')
plt.tight_layout(); plt.show()

def upper_vec(kx, ky, D=0.0):
    w, v = np.linalg.eigh(H(kx, ky, D))
    return v[:, 1]
def berry(E, valley, D=0.0, n=240):
    phis, r = contour(E, valley, D, n)
    us = [upper_vec(*(valley + ri * np.array([np.cos(p), np.sin(p)])), D) for p, ri in zip(phis, r)]
    prod = np.prod([np.vdot(us[i], us[(i + 1) % n]) for i in range(n)])
    return -np.angle(prod)
for E in (0.3, 1.0):
    print(f'Delta = 0, E = {E}: |Berry phase| / pi = {abs(berry(E, K)) / np.pi:.4f} at K, {abs(berry(E, Kp)) / np.pi:.4f} at K\' (its sign is rounding: +pi = -pi)')
for D, E in [(0.01, 0.3), (0.3, 0.6)]:
    print(f'Delta = {D}, E = {E}: Berry phase / pi = {berry(E, K, D) / np.pi:+.4f} at K, {berry(E, Kp, D) / np.pi:+.4f} at K\'  (pi(1 - Delta/E) = {1 - D / E:.4f})')
print(f'mass for Delta = 0.3 eV: m* = Delta / v_F^2 = {0.3 * e / vF ** 2 / m_e:.4f} m_e')

## Check the lab's answers

These asserts confirm the numbers in the lab's "Predict first" answers.

In [ ]:
# Q1: E = +-3t = +-8.1 eV at Gamma; bonding (A, B in phase) below, antibonding above
assert np.allclose(bands(*G), (-8.1, 8.1))
w, v = np.linalg.eigh(H(*G)); assert np.isclose(v[0, 0] * v[1, 0], 0.5) and np.isclose(v[0, 1] * v[1, 1], -0.5)
# Q2: g(K) = 0 for any t, since the phases are 1, e^{2pi i/3}, e^{-2pi i/3}
assert abs(g(*K)) < 1e-12 and np.allclose(sorted(np.round(np.degrees(np.angle(np.exp(1j * DEL @ K))))), [-120, 0, 120])
# Q3: |g(M)| = 1, E = +-t
assert np.isclose(abs(g(*M)), 1) and np.allclose(bands(*M), (-2.7, 2.7))
# Q4: E_F = 0.3 eV: k_F = 0.52 1/nm, n = 8.7e12 cm^-2, one electron per ~440 atoms
assert round(0.3 / hv, 2) == 0.52 and abs(0.3 ** 2 / (np.pi * hv ** 2) * 1e14 - 8.7e12) < 0.05e12 and abs(density(0.3) * 1e14 / 8.7e12 - 1) < 0.02
assert round(2 / cell / (0.3 ** 2 / (np.pi * hv ** 2)), -1) == 440
# Q5: at E_F = t the DOS diverges and n = 9.5e14 cm^-2, a quarter of an electron per atom
assert abs(density(2.7) * 1e14 / 9.5e14 - 1) < 0.01 and abs(density(2.7) / (2 / cell) - 0.25) < 0.003
# Q6: the DOS vanishes linearly at E = 0: the exact DOS halves when E halves, and the histogram's bins nearest 0 are tiny
assert abs(rho_exact(0.05) / rho_exact(0.1) - 0.5) < 0.005 and abs(rho_exact(0.1) / (2 * 0.1 / (np.pi * hv ** 2) * cell) - 1) < 0.01
assert np.max(rho_hist[np.abs(Ec) < 0.025]) < 0.005
# Q7: at 0.1 eV the largest and smallest radii differ by 1.2% and the mean matches E/hbar v_F to 0.01%; at 1 eV 14% and +1%
phis, r = contour(0.1); assert abs(r.max() / r.min() - 1 - 0.012) < 0.001 and abs(r.mean() * hv / 0.1 - 1) < 0.0005
phis, r = contour(1.0); assert abs(r.max() / r.min() - 1.14) < 0.005 and abs(r.mean() * hv - 1.01) < 0.002
# Q8: Delta = 0.3 eV: m* = 0.069 m_e, Berry phase pi/2 at E = 0.6 eV, opposite in the two valleys
assert round(0.3 * e / vF ** 2 / m_e, 3) == 0.069 and abs(abs(berry(0.6, K, 0.3)) - np.pi / 2) < 1e-3 and berry(0.6, K, 0.3) * berry(0.6, Kp, 0.3) < 0
print('all the lab\'s answers check out')

## Things to try

1. **Strain.** Make one of the three hoppings $t_1\ne t_2=t_3$. The Dirac points move away from K and K′; when $t_1>2t$ they meet and annihilate, and a gap opens.
2. **Next-nearest neighbours.** Add $-t'(|g|^2-3)$ to both bands. Which features of the DOS move, and are electrons and holes still mirror images?
3. **Zigzag ribbons.** Make the lattice finite in one direction and diagonalise the ribbon Hamiltonian for each $k_x$. Look for the flat band of edge states between K and K′.
4. **Haldane's model.** Give the second-neighbour hopping a phase $e^{\pm i\phi}$. With $\Delta=0$ a gap opens in which the Berry phases of the two valleys no longer cancel: a Chern insulator.